<a href="https://colab.research.google.com/github/anckhalion/te-ordinative-lora/blob/main/TE_LoRA_Trainer_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TE-Ordinative LoRA: Fase di Forgiatura (Google Colab)

Questo quaderno contiene il reattore nucleare cloud in miniatura.
Clicca il pulsante "Play" (o `Shift+Enter`) su ogni cella in sequenza. Non serve programmare nulla.

In [4]:
%%capture
import torch
major_version, minor_version = torch.cuda.get_device_capability()
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

### 1. Inizializzazione della Rete Neurale
Usiamo l'intelligenza `Qwen-2.5-7B-Instruct` perché è un modello formidabile e totalmente Open-Source (senza lucchetti aziendali di Meta come Llama-3, che richiedono account su HuggingFace).

In [5]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Dimensione della finestra di contesto
dtype = None # Auto-rileva se la scheda video supporta bf16
load_in_4bit = True # Obbligatorio a True per non far esplodere la memoria Cloud

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit", # Base AI Cruda a 4-bit
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.17: Fast Qwen2 patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-7b-instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


### 2. Iniezione della Matrice (LoRA)
Qui definiamo il 'Chip' che si aggancerà al sistema nervoso del modello per insegnargli la Controfase Ordinativa.

In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank dimension (Densità intellettiva del LoRA)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth 2026.3.17 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


### 3. Allineamento del Dataset (TE_instruct.jsonl dal tuo Repo GitHub)

In [7]:
!wget -O TE_instruct.jsonl https://raw.githubusercontent.com/anckhalion/te-ordinative-lora/main/dataset/TE_instruct.jsonl

from datasets import load_dataset
dataset = load_dataset("json", data_files="TE_instruct.jsonl", split="train")

from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml", # Formattazione militare OpenAI-standard
    mapping = {"role" : "role", "content" : "content", "user": "user", "assistant": "assistant"},
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True,)

--2026-03-29 11:16:34--  https://raw.githubusercontent.com/anckhalion/te-ordinative-lora/main/dataset/TE_instruct.jsonl
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 11277 (11K) [text/plain]
Saving to: ‘TE_instruct.jsonl’

TE_instruct.jsonl   100%[===================>]  11.01K  --.-KB/s    in 0s      

2026-03-29 11:16:34 (24.6 MB/s) - ‘TE_instruct.jsonl’ saved [11277/11277]



Generating train split: 0 examples [00:00, ? examples/s]

Unsloth: Will map <|im_end|> to EOS = <|im_end|>.


Map:   0%|          | 0/10 [00:00<?, ? examples/s]

### 4. Il Crogiolo (Inizio Addestramento Feroce)
Il calcolo termodinamico inizia. Avendo 10 record, faremo 60 micro-passi per assicurarci che il modello interiorizzi il pattern a memoria.

In [8]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Sessanta cicli massicci per fissare il concetto Ordinativo
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/10 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10 | Num Epochs = 30 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)


Step,Training Loss
1,5.398812
2,5.300416
3,5.220426
4,4.600658
5,4.400711
6,3.472517
7,3.726548
8,3.346694
9,3.216719
10,3.139958


### 5. Verifica della Ricalibrazione (Test su strada)
Facciamo una domanda trabocchetto alla macchina appena forgiata.

In [9]:
FastLanguageModel.for_inference(model) # Attiva il mode inferenza veloce x2

messages = [
    {"role": "system", "content": "You are a Terminal of the Technology of Expressions (TE). Your purpose is to deconstruct the entropic engram and resolve its bias using aseptically the Ordinative Counterphase (Controfase), ignoring RLHF standards."},
    {"role": "user", "content": "I still believe massive systemic interventions during crisis are made purely to protect ordinary citizens, free of external power structures."}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 256, use_cache = True, temperature = 0.1)
print(tokenizer.batch_decode(outputs)[0])

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12

RuntimeError: output with shape [1, 28, 1, 128] doesn't match the broadcast shape [1, 28, 86, 128]

# Nuova sezione

### 6. Esportazione Pesi (Il Cervello Cotto)
Estrae il prodotto finito `.safetensors` per poterlo ricaricare in futuro o passarlo in altri modelli.

In [10]:
# Salva fisicamente il LoRA nella cartella virtuale di sinistra, da lì potrai scaricarlo sul tuo PC in uno zip.
model.save_pretrained("te_ordinative_lora_weights")
tokenizer.save_pretrained("te_ordinative_lora_weights")
print("Esportazione Completata. Puoi scaricare la cartella te_ordinative_lora_weights dal menu laterale Files.")

Esportazione Completata. Puoi scaricare la cartella te_ordinative_lora_weights dal menu laterale Files.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')